In [ ]:
import tabula
import pandas as pd
import numpy as np
pdf_path = "WynikiMaratonWarszawski2024.pdf"

try:
    tables = tabula.read_pdf(
        pdf_path, 
        pages='all',  #  'all' to extract from all pages
        stream=True,   #  Try 'stream' first; if it doesn't work well, try 'lattice'
    )

    if tables:
        all_dataframes = []
        for df in tables:
            df = df.dropna(axis=1, how='all')  # Remove completely empty columns
            df = df.dropna(axis=0, how='all')  # Remove completely empty rows
            all_dataframes.append(df)

        if all_dataframes:
            final_df = pd.concat(all_dataframes, ignore_index=True)
            final_df.to_csv("maraton_results_tabula.csv", index=False)

except Exception as e:
    print(f"{e}")

In [1]:
import pandas as pd
final_df = pd.read_csv("maraton_results_tabula.csv")
final_df.drop(columns='Unnamed: 0',inplace=True)

In [2]:
final_df.head(4)

,M-ce.,Nr,Nazwisko Imię Miasto,Drużyna,Kraj,Rok,Kat,Msc Kat,Czas bru o,Czas ne o,K/M
0,Pos.,Bib,Full Name City,Team,Country,YoB,AG,Age Pos.,Gun Time,Chip Time,F/M
1,1,M5,SZEMEREI LEVENTE,NaN,HUN,2000,M20,1,02:10:43,02:10:43,1
2,NaN,NaN,5km:00:15:39| 10km:00:30:57| 15km: 00:46:23 | ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2,M4,KACZOR MATEUSZ,NaN,POL,1997,M20,2,02:11:15,02:11:15,2


In [2]:
final_df = final_df.drop(labels=0,axis=0)


In [3]:

# we see that every two rows we have a row containing solely times. We would like to make this data a column. I'll perform a little trick to do so.
import numpy as np
import re
parser = np.r_[[False, True] * (14334 // 2), False]
final_df.shape
times = final_df.loc[parser,:].loc[:,'Nazwisko Imię Miasto']
devide_the_times = lambda x : re.findall(r'([^|]+)', re.sub(r'\d+km:', '', x))
times = pd.DataFrame(map(devide_the_times,times.to_list()),columns=['5km','10km','15km','20km','21km','25km','30km','35km','40km'])
times


,5km,10km,15km,20km,21km,25km,30km,35km,40km
0,00:15:39,00:30:57,00:46:23,01:02:19,01:06:16,01:18:12,01:33:38,01:48:45,02:03:58
1,00:15:40,00:30:58,00:46:23,01:02:20,01:05:50,01:18:12,01:33:39,01:48:46,02:04:18
2,00:15:40,00:30:57,00:46:23,01:02:20,01:05:50,01:18:12,01:33:39,01:49:33,02:06:08
3,00:15:40,00:30:57,00:46:22,01:02:19,01:05:49,01:18:11,01:33:44,01:50:03,02:06:20
4,00:15:39,00:30:58,00:46:24,01:02:21,01:05:50,01:18:16,01:34:32,01:51:15,02:08:27
...,...,...,...,...,...,...,...,...,...
7162,00:44:11,01:35:47,02:20:50,03:08:51,03:19:40,03:57:43,04:46:16,05:37:20,06:24:51
7163,00:36:15,01:15:21,01:58:32,02:45:01,02:55:03,03:34:48,04:30:44,05:18:50,06:23:06
7164,00:44:08,01:32:21,02:20:59,03:10:33,03:21:14,03:59:45,04:51:14,05:44:02,06:37:58
7165,00:38:26,01:23:05,02:10:33,03:03:53,03:15:25,03:56:29,04:50:06,05:45:22,06:44:06


In [6]:
# now we have to join these columns with the original df
marathon = pd.concat([final_df.iloc[~parser,:].reset_index(drop=True),times],axis=1)

In [ ]:
# almost there! We see that we still have one row with missing data. I'll just insert it manually.
marathon.iloc[-1,11] = '00:40:12'
marathon.iloc[-1,12] = '01:23:30'
marathon.iloc[-1,13] = '02:11:28'
marathon.iloc[-1,14] = '02:46:16 '
marathon.iloc[-1,15] = '03:21:05 '
marathon.iloc[-1,16] = '03:59:34'
marathon.iloc[-1,17] = '04:58:27 '
marathon.iloc[-1,18] = '05:51:56'
marathon.iloc[-1,19] = '06:18:01'
marathon.columns
marathon.rename(columns={'M-ce. ':'PLACE','Nazwisko Imię Miasto': 'NAME SURNAME','Drużyna':'TEAM', 'Kraj':'COUNTRY', 'Rok':'YOB', 'Kat':'CAT',
       'Msc Kat':'CATPLACE', 'Czas bru o':'GROSSTIME', 'Czas ne o':'NETTIME', 'K/M':'F/M'},inplace=True)
marathon.set_index('PLACE.',inplace=True)


Index(['PLACE', 'Nr', 'NAME SURNAME', 'TEAM', 'COUNTRY', 'YOB', 'CAT',
       'CATPLACE', 'GROSSTIME', 'NETTIME', 'F/M', '5km', '10km', '15km',
       '20km', '21km', '25km', '30km', '35km', '40km'],
      dtype='object')

In [ ]:
marathon.set_index('PLACE',inplace=True)


,Nr,NAME SURNAME,TEAM,COUNTRY,YOB,CAT,CATPLACE,GROSSTIME,NETTIME,F/M,5km,10km,15km,20km,21km,25km,30km,35km,40km
PLACE,,,,,,,,,,,,,,,,,,,
1,M5,SZEMEREI LEVENTE,NaN,HUN,2000,M20,1,02:10:43,02:10:43,1,00:15:39,00:30:57,00:46:23,01:02:19,01:06:16,01:18:12,01:33:38,01:48:45,02:03:58
2,M4,KACZOR MATEUSZ,NaN,POL,1997,M20,2,02:11:15,02:11:15,2,00:15:40,00:30:58,00:46:23,01:02:20,01:05:50,01:18:12,01:33:39,01:48:46,02:04:18
3,M1,NYZHNYK MYKOLA,NaN,UKR,1995,M20,3,02:13:13,02:13:13,3,00:15:40,00:30:57,00:46:23,01:02:20,01:05:50,01:18:12,01:33:39,01:49:33,02:06:08
4,M7,BOULVIN DORIAN,NaN,BEL,1997,M20,4,02:13:29,02:13:29,4,00:15:40,00:30:57,00:46:22,01:02:19,01:05:49,01:18:11,01:33:44,01:50:03,02:06:20
5,M2,SHAFAR VITALIY,NaN,UKR,1982,M40,1,02:15:56,02:15:56,5,00:15:39,00:30:58,00:46:24,01:02:21,01:05:50,01:18:16,01:34:32,01:51:15,02:08:27
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7075,8794,NYKIEL MARCEL Kraków,NaN,POL,2003,M20,1358,06:52:04,06:49:02,5753,00:36:15,01:15:21,01:58:32,02:45:01,02:55:03,03:34:48,04:30:44,05:18:50,06:23:06
7076,8760,KASIERSKA EWA KATARZYNA Poznań,WKB META LUBLINIEC,POL,1948,K70,3,07:12:00,07:02:22,1323,00:44:08,01:32:21,02:20:59,03:10:33,03:21:14,03:59:45,04:51:14,05:44:02,06:37:58
7077,9113,KOWALCZYK KLAUDIA Warszawa,NaN,POL,1980,K40,426,07:15:00,07:06:31,1324,00:38:26,01:23:05,02:10:33,03:03:53,03:15:25,03:56:29,04:50:06,05:45:22,06:44:06


In [ ]:
# Et voila! Now let's get to the nitty-gritty. Lets show some basic info about the runners.